# FSOT-2.1-Neural — Emotions EEG + Morse Chew/Query

**Theory authority:** [FSOT-2.1-Lean](https://github.com/dappalumbo91/FSOT-2.1-Lean)  
**License (this notebook + code):** Apache-2.0  
**Not a medical device.**

This notebook:
1. Loads **Kaggle EEG emotion features** (`emotions.csv` — POSITIVE / NEUTRAL / NEGATIVE)
2. Loads **Allen NWB** ephys stats (if staged)
3. Derives FSOT lesion/drive priors
4. Chews local FSOT literature and answers queries via **ITU Morse + reservoir retrieval**

### Dataset license note
The Kaggle EEG emotion CSV is **third-party**. Do **not** re-upload that CSV into a new Kaggle dataset unless the original license allows it.  
Publish **this notebook + FSOT-2.1-Neural code** under Apache-2.0, and *link* the original Kaggle dataset as an input.

In [ ]:
import sys
from pathlib import Path

# Point at repo root (edit if running elsewhere)
ROOT = Path(r"I:\fsot nuron")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT", ROOT, "exists", ROOT.is_dir())

## 1. Kaggle emotions EEG features

In [ ]:
from fsot_nuron.emotions_eeg import find_emotions_csv, compute_emotions_stats, emotion_conditioned_drive
import json

csv_path = find_emotions_csv()
print("emotions.csv:", csv_path)

stats = compute_emotions_stats()
print("samples", stats["n_samples"], "features_used", stats["n_features_used"])
print("labels", stats["labels"])
print("energy_cv", stats["global"]["sample_energy_cv"])
print("emotion templates:")
print(json.dumps(stats["emotion_lesion_templates"], indent=2))

In [ ]:
import matplotlib.pyplot as plt

for lab in ["NEGATIVE", "NEUTRAL", "POSITIVE"]:
    d = emotion_conditioned_drive(lab, length=200, seed=0)
    plt.plot(d.numpy(), label=lab, alpha=0.85)
plt.legend()
plt.title("Emotion-conditioned FSOT drive (from Kaggle feature stats)")
plt.xlabel("t (steps)")
plt.ylabel("stim")
plt.show()

## 2. Allen NWB + blended real-signal bundle

In [ ]:
from fsot_nuron.emotions_eeg import integrate_with_eeg_bundle

bundle = integrate_with_eeg_bundle()
print("bundle ok", bundle.get("ok"))
print("allen nwb", bundle.get("allen_nwb", {}).get("ok"), bundle.get("allen_nwb", {}).get("aggregate"))
print("kaggle emotions", bundle.get("kaggle_emotions", {}).get("ok"))
print("derived priors", bundle.get("derived_priors"))

## 3. PD / emotion failure boundary probe (optional)

In [ ]:
from fsot_nuron.failure_boundaries import probe_failure_mode

r = probe_failure_mode("PD_rate_irregularity", n_units=32, steps=400, device="cpu", use_eeg=True)
print("breached", r["lesioned"]["breached"], "recovered", r["wire_around"]["recovered_envelope"])
print("rates H/L/W",
      r["healthy"]["population"]["mean_firing_rate_Hz"],
      r["lesioned"]["population"]["mean_firing_rate_Hz"],
      r["wire_around"]["population"]["mean_firing_rate_Hz"])
print("auto wire", r["wire_around"].get("strategy"), r["wire_around"].get("actions"))

## 4. Literature chew + Morse query (LLM-style use)

In [ ]:
from fsot_nuron.literature_chew import LiteratureMind

mind = LiteratureMind(n_units=16, device="cpu")
if not mind.load_memory() or len(mind.memory) < 5:
    print("Chewing literature (first run)...")
    print(mind.chew_documents(max_chunks=24))
else:
    print("Loaded memory", len(mind.memory))

for q in [
    "What is Fluid Spacetime Omni-Theory?",
    "How does Morse trinary encode language?",
    "What are zero free parameters?",
]:
    print("\n=== Q:", q)
    ans = mind.query(q, top_k=3)
    print(ans.get("answer", ans))

## 5. Publishing guidance (Kaggle vs GitHub)

| Channel | What to publish | License |
|---------|-----------------|--------|
| **GitHub `FSOT-2.1-Neural`** | Code, notebook, docs, codon/Morse tables | **Apache-2.0** (matches your FSOT repos) |
| **Kaggle Notebook** | This notebook as a *kernel* that **reads** the existing emotions dataset | Apache-2.0 for notebook code |
| **Kaggle Dataset** | Only if **you** create *derived* FSOT outputs (stats JSON, reports) — not a silent re-upload of `emotions.csv` | Check original Kaggle dataset license first |

**Recommendation:**  
- **Apache-2.0** for FSOT code (aligns with Lean/GPU repos; patent grant useful for systems work).  
- MIT is fine for tiny scripts but Apache-2.0 is better as the main stack license.  
- Put the **theory pin** on GitHub (FSOT-2.1-Lean). Put a **runnable demo** notebook on Kaggle that depends on the public emotions EEG dataset as input.